# M2.2 Export my model to TFLite (FP32 + full INT8)
Ultralytics only exports TFLite on Linux/macOS, so this runs in Colab (a CPU runtime is enough).

Colab's own Python (3.13 + TensorFlow 2.21) does not match what the TFLite converter needs, so cell 1 builds a
separate **Python 3.12** environment (run with `MPLBACKEND=Agg`, because Colab's notebook plot backend does not exist there) with pinned, compatible versions. Takes ~3 minutes.

1. On the laptop: `python scripts/make_export_bundle.py`
2. Runtime -> Run all, upload `export_bundle.zip` when asked
3. Send `m2_tflite.zip` (it must contain `models/best_int8.tflite`)

In [ ]:
# 1) isolated Python 3.12 env with versions that work together
!pip -q install uv
!uv venv -q -p 3.12 /content/py312
!uv pip install -q --python /content/py312/bin/python     "ultralytics==8.3.253" "tensorflow==2.19.0" "tf_keras==2.19.0" "onnx==1.17.0" "onnx2tf==1.27.10"     "onnxslim==0.1.71" "onnxruntime==1.22.0" "sng4onnx==1.0.4" "onnx_graphsurgeon==0.5.8" "ai-edge-litert==1.2.0"     "protobuf==5.29.5" "numpy<2.2" "opencv-python-headless"
!MPLBACKEND=Agg /content/py312/bin/python -c "import tensorflow as tf, tf_keras, onnx, onnx2tf, ultralytics; print('tf', tf.__version__, '| onnx', onnx.__version__, '| onnx2tf', onnx2tf.__version__, '| ultralytics', ultralytics.__version__)"

In [ ]:
# 2) upload export_bundle.zip
from google.colab import files
up = files.upload()
!rm -rf /content/bundle && mkdir -p /content/bundle && unzip -q export_bundle.zip -d /content/bundle
%cd /content/bundle

In [ ]:
# 3) export: INT8 calibration = 200 images from MY train split (never val/test)
#    YOLO_AUTOINSTALL=false: never let Ultralytics change packages behind my back
!MPLBACKEND=Agg YOLO_AUTOINSTALL=false YOLO_OFFLINE=true /content/py312/bin/python scripts/export.py --format tflite --int8 --calib-n 200
!ls -la models/ && cat evidence/M2/export.json

In [ ]:
# 4) check the INT8 model loads and runs before downloading
!MPLBACKEND=Agg /content/py312/bin/python -c "import numpy as np; from ai_edge_litert.interpreter import Interpreter; it=Interpreter('models/best_int8.tflite'); it.allocate_tensors(); i=it.get_input_details()[0]; o=it.get_output_details()[0]; it.set_tensor(i['index'], np.zeros(i['shape'], i['dtype'])); it.invoke(); print('INT8 ok', i['dtype'], i['shape'], '->', o['dtype'], it.get_tensor(o['index']).shape)"

In [ ]:
import os
assert os.path.exists('models/best_int8.tflite'), 'export failed - read the error in cell 3, do not download'
!zip -q m2_tflite.zip models/*.tflite evidence/M2/export.json runs/calib/calib.txt runs/calib/calib_info.json
!unzip -l m2_tflite.zip
files.download('m2_tflite.zip')